In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.models.nn_pyomo_base import NeuralODEPyomo
from src.models.direct_collocation import BarycentricInterpolation

## 1. Create dataset of bioreactor state vectors

In [ ]:
DATA_PATH = "../datasets/bioreactor_sim_by_noise/noise_level_01_eps0.01.csv"
STATE_COLS = ['G', 'O', 'X', 'Xd', 'P', 'L', 'CO2']

df = pd.read_csv(DATA_PATH)
batch_0 = df[df['Batch'] == 'Batch_0'].reset_index(drop=True)

train_grid = batch_0['Time'].values          # (30,)
Y_obs = batch_0[STATE_COLS].values           # (30, 7)
end_time = float(train_grid[-1])

print(f"Time grid shape:  {train_grid.shape}")
print(f"Y_obs shape:      {Y_obs.shape}")
print(f"End time:         {end_time:.4f}")
print()
print(f"{'State':<8} {'Mean':>8} {'Min':>8} {'Max':>8}")
print("-" * 36)
for i, col in enumerate(STATE_COLS):
    print(f"{col:<8} {Y_obs[:, i].mean():>8.3f} {Y_obs[:, i].min():>8.3f} {Y_obs[:, i].max():>8.3f}")

# Plot all 7 state trajectories
fig, axes = plt.subplots(4, 2, figsize=(13, 11))
axes = axes.flatten()
for i, col in enumerate(STATE_COLS):
    axes[i].scatter(train_grid, Y_obs[:, i], color='darkorange', marker='o', s=25, alpha=0.85)
    axes[i].set_title(col, fontsize=13)
    axes[i].set_xlabel('Time (h)')
    axes[i].set_ylabel('Concentration (g/L)')
    axes[i].grid(True, linestyle='--', color='lightgrey', alpha=0.8)
axes[7].set_visible(False)
plt.suptitle("Bioreactor Batch_0 State Trajectories  (noise level ε = 0.01)", fontsize=14)
plt.tight_layout()

## 2. Optimise pyomo model using IPOPT (train the neural ODE)

In [ ]:
layer_sizes = [7, 32, 7]
num_res_eval_nodes = 50

solver_options_dc = {
    "max_iter": 500,
    "nlp_scaling_method": "gradient-based",
    "mu_strategy": "adaptive",
    "tol": 1e-6,
    "acceptable_tol": 1e-5,
    "acceptable_iter": 10
}

solver_options_irrdc = {
    "max_iter": 1000,
    "nlp_scaling_method": "gradient-based",
    "mu_strategy": "adaptive",
    "tol": 1e-6,
    "acceptable_tol": 1e-5,
    "acceptable_iter": 10
}

pyo_model_dc = NeuralODEPyomo(
    Y_obs, layer_sizes, end_time,
    state_lower_bound=-1000,
    state_upper_bound=1000,
    param_lower_bound=-100,
    param_upper_bound=100,
    lambda_reg=1e-4,
    transcription_method='dc'
)
pyo_model_dc.solve_model(solver_options_dc)

In [ ]:
pyo_model_irrdc = NeuralODEPyomo(
    Y_obs, layer_sizes, end_time,
    state_lower_bound=-1000,
    state_upper_bound=1000,
    param_lower_bound=-100,
    param_upper_bound=100,
    lambda_reg=1e-4,
    transcription_method='irrdc',
    rho_reg=5,
    num_res_eval_nodes=num_res_eval_nodes
)
pyo_model_irrdc.solve_model(solver_options_irrdc)

## 3. Use the trained model as RHS of an ODE system

In [ ]:
y0 = Y_obs[0, :]                                # initial condition from Batch_0
dt = float(train_grid[1] - train_grid[0])       # equidistant step size

predicted_train_dc = pyo_model_dc.get_predicted_trajectory(
    y0, train_grid, rtol=1e-7, atol=1e-9, max_step=dt
)
predicted_train_irrdc = pyo_model_irrdc.get_predicted_trajectory(
    y0, train_grid, rtol=1e-7, atol=1e-9, max_step=dt
)

print(f"DC    predicted trajectory shape: {predicted_train_dc.shape}")
print(f"IRRDC predicted trajectory shape: {predicted_train_irrdc.shape}")

## 4. Plot predicted trajectories over train and test domains

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(13, 11))
axes = axes.flatten()

for i, col in enumerate(STATE_COLS):
    ax = axes[i]
    ax.scatter(train_grid, Y_obs[:, i],
               color='darkorange', marker='o', s=30, alpha=0.85, label='Observed (noisy)', zorder=3)
    ax.plot(train_grid, predicted_train_dc[:, i],
            color='firebrick', linewidth=1.8, label='DC')
    ax.plot(train_grid, predicted_train_irrdc[:, i],
            color='forestgreen', linewidth=1.8, label='IRR-DC')
    ax.set_title(col, fontsize=13)
    ax.set_xlabel('Time (h)')
    ax.set_ylabel('Concentration (g/L)')
    ax.grid(True, linestyle='--', color='lightgrey', alpha=0.8)
    if i == 0:
        ax.legend(fontsize=9)

axes[7].set_visible(False)
plt.suptitle(
    "Bioreactor Batch_0: Predicted vs Observed Trajectories\n"
    "(ε = 0.01, DC vs IRR-DC, single hidden layer 7→32→7)",
    fontsize=13
)
plt.tight_layout()

## 5. Plot ODE residual in training domain

In [ ]:
def compute_continuous_residual(pyo_model, num_eval_nodes):
    """
    Compute r(t) = dY*/dt - NN(Y*(t)) on a dense equidistant grid via
    barycentric interpolation/differentiation of the trained collocation solution.
    Works for both 'dc' and 'irrdc' models.
    Returns: t_eval (num_eval_nodes,), residual (num_eval_nodes, state_dim)
    """
    N = pyo_model.num_colloc_nodes
    bary = BarycentricInterpolation(
        0, pyo_model.end_time, N,
        transcription_method='irrdc',
        num_res_eval_nodes=num_eval_nodes
    )
    L     = bary.L_res                           # (num_eval_nodes, N)
    D     = bary.D_res                           # (num_eval_nodes, N)
    t_eval = bary.res_eval_grid.astype(float)

    Y_star = pyo_model.pyomo_var_to_numpy(
        pyo_model.model.Y_star, (N, pyo_model.state_dim)
    )
    Ws = pyo_model.convert_weights()
    bs = pyo_model.convert_biases()

    ode_lhs = D @ Y_star                         # d(interpolant)/dt at eval nodes
    Y_eval  = L @ Y_star                         # interpolant values at eval nodes
    nn_out  = np.array([
        pyo_model.nn_prediction(0, Y_eval[i], Ws, bs)
        for i in range(num_eval_nodes)
    ])

    return t_eval, ode_lhs - nn_out


num_eval_nodes = 500
skip = 10   # skip boundary points where barycentric interpolation can spike

t_dc,    res_dc    = compute_continuous_residual(pyo_model_dc,    num_eval_nodes)
t_irrdc, res_irrdc = compute_continuous_residual(pyo_model_irrdc, num_eval_nodes)

t_dc     = t_dc[skip:]
res_dc   = res_dc[skip:]
t_irrdc  = t_irrdc[skip:]
res_irrdc = res_irrdc[skip:]

fig, axes = plt.subplots(4, 2, figsize=(13, 11))
axes = axes.flatten()

for i, col in enumerate(STATE_COLS):
    ax = axes[i]
    ax.plot(t_dc,    np.abs(res_dc[:, i]),
            color='firebrick',   linewidth=1.0, label='DC')
    ax.plot(t_irrdc, np.abs(res_irrdc[:, i]),
            color='forestgreen', linewidth=1.0,
            label=fr'IRR-DC ($\rho$={pyo_model_irrdc.rho_reg})')
    ax.set_title(col, fontsize=13)
    ax.set_xlabel('t (h)')
    ax.set_ylabel(r'$|r(t)|$')
    ax.grid(True, linestyle='--', color='lightgrey', alpha=0.8)
    if i == 0:
        ax.legend(fontsize=9)

axes[7].set_visible(False)
plt.suptitle(
    r'ODE Residual in Training Domain:  '
    r'$\mathbf{r}(t) = \dot{\mathbf{Y}}(t) - \mathbf{F}_{\boldsymbol{\theta}}(\mathbf{Y}(t))$',
    fontsize=13
)
plt.tight_layout()